# Case 5 · Replace the CEO after a bad year? (Part 2)
**BUS-G 492 · Causal decisions with AI**

In [ ]:
# --- Setup: run this cell first -----------------------------------------------
import os
DATA_URL = "https://raw.githubusercontent.com/boyoung-seo/492-coding/main/cases/case5_natural_ceo/data/"
for candidate in ["data/", "cases/case5_natural_ceo/data/", "../../../cases/case5_natural_ceo/data/", "../../../../cases/case5_natural_ceo/data/"]:
    if os.path.exists(candidate + "company_years.csv"):
        DATA = candidate
        break
else:
    DATA = DATA_URL
print("Reading data from:", DATA)

## Stage 5 · Something you didn't know

> **From:** Omar Haddad, Research Associate
> **Re:** why CEOs leave
>
> I coded the **reason** for every CEO change in our 600-company dataset from news archives: `ceo_departure_reasons.csv`.
>
> Most changes happen because the **board pushed the CEO out after a bad year**. Some are planned retirements. And about 1% of CEOs a year leave because of **death or sudden serious illness**, which has nothing to do with how the company was doing.

### Load the data

This is a new notebook, so nothing from Part 1 is loaded yet. Run this cell to load the Part 1 data again, then the next cell for the new files.

In [ ]:
import statsmodels.formula.api as smf
import pandas as pd, numpy as np
df = pd.read_csv(DATA + "company_years.csv")
port = pd.read_csv(DATA + "portfolio_companies.csv")
print(df.shape, port.shape)
df.head()

In [ ]:
reasons = pd.read_csv(DATA + "ceo_departure_reasons.csv")
reasons.departure_reason.value_counts()

## Your tasks

Use AI again, but this time **tell it the whole story**: how the data were generated, and what the new information is. Each step ends with a table: fill it in (double-click the cell to type) before moving on.

### Step 1 · Why did the CEO leave?

Merge the departure reasons onto the company data (on `firm_id` and `year`; years without a change get "no change"). For each reason, compute the ROA change from the year before to the year after:

| Departure reason | ROA change (points) | N |
|---|---|---|
| Resigned under board pressure | | |
| Planned retirement | | |
| **Death or sudden illness (ETT)** | | |
| No change | | |

Which row is the natural experiment, and what does it say a new CEO does?

In [ ]:
df = df.sort_values(["firm_id", "year"])
df["prev"] = df.groupby("firm_id").roa_pct.shift(1)
df["next"] = df.groupby("firm_id").roa_pct.shift(-1)
df["change"] = df.next - df.prev
d = df.merge(reasons, on=["firm_id", "year"], how="left")
d["departure_reason"] = d.departure_reason.fillna("no change")
# Your code here


### Step 2 · A bad year, same CEO

Find company-years where ROA the **year before** was at least **3 points below that company's own average**. What happened next?

| After a bad year | ROA change (points) |
|---|---|
| CEO **kept** | |
| CEO **replaced** | |

How much of the rebound happens without a new CEO?

In [ ]:
df["firm_avg"] = df.groupby("firm_id").roa_pct.transform("mean")
bad = df[(df.firm_avg - df.prev) >= 3].dropna(subset=["change"])
# Your code here


### Step 3 · One bad year or a weak CEO? Our 20 companies

For each portfolio company, compare ROA **under the current CEO** (from the year they started up to 2024) with ROA **before** that CEO arrived. The code below builds the table. List the **five** companies with the most negative difference, then look at their year-by-year ROA before deciding.

| Company | CEO since | ROA before CEO | ROA under CEO (to 2024) | Difference | Replace? |
|---|---|---|---|---|---|
| | | | | | |
| | | | | | |
| | | | | | |
| | | | | | |
| | | | | | |

In [ ]:
rows = []
for f, g in port.groupby("firm_id"):
    start = g[g.new_ceo_this_year == 1].year.max()
    before = g[g.year < start].roa_pct.mean()
    under = g[(g.year >= start) & (g.year < 2025)].roa_pct.mean()
    rows.append((f, int(start), round(before, 1), round(under, 1), round(under - before, 1)))
table = pd.DataFrame(rows, columns=["firm", "CEO since", "ROA before", "ROA under (to 2024)", "difference"]).sort_values("difference")
table
# Your code here: e.g. port.pivot(index="firm_id", columns="year", values="roa_pct").loc[["P09", "P04"]]


### Decide again

In [ ]:
TEAM = ""                  # e.g. "Team 3: Ana & Ben"  (use the same name every time)
FIRMS_TO_REPLACE = None    # company numbers separated by commas ("4, 9" = P04 and P09); "0" = replace no one  (e.g. "4, 9")
PREDICTED_PROFIT = None    # in dollars, as a number, e.g. 150000
JUSTIFICATION = """
Three sentences for the decision maker: what you found, what you recommend, and why.
"""

# Stops with a message if anything is still blank, so you don't submit an empty field.
blank = [name for name, value in [("TEAM", TEAM), ("FIRMS_TO_REPLACE", FIRMS_TO_REPLACE), ("PREDICTED_PROFIT", PREDICTED_PROFIT)] if value in (None, "")]
if blank:
    raise ValueError("Fill these in before submitting: " + ", ".join(blank))
PREDICTED_PROFIT = float(PREDICTED_PROFIT)   # must be a number
print("Enter exactly these values in the submission form:")
print()
print(f"team:              {TEAM}")
print("case:              5")
print("round:             2  (Part II)")
print(f"decision:          {FIRMS_TO_REPLACE}")
print(f"predicted_profit:  {PREDICTED_PROFIT:.0f}")

## Submit (round 2)

**[Open the submission form](https://docs.google.com/forms/d/e/1FAIpQLSf-87eDsEHRVat_a4le91sC7bfZFel9fAHNmMtHVuGJzqaHOQ/viewform)** and enter the values printed above, exactly as shown.

- **Predicted profit:** a number such as `150000` (`$150,000` and `150K` also work, but not words). The decision is your company numbers separated by commas, e.g. `4, 9` for P04 and P09. Enter `0` to replace no one.
- **Case:** `5`.  **Round:** `2` (Part II).
- **Submissions are final.** You can't change an answer after you submit, so check the printed values first. If your team submits more than once for the same round, only the **first** submission counts.
- Use the **same team name** in every round.

## Reflection (submit)

1. Your Part 1 analysis may have fit the data very well. Why was it still the wrong tool for this decision?
2. In one sentence: what was the difference between **predicting** the outcome and **deciding** what to do?
3. Name one other business decision where the same trap could appear.